# KlippEye: train the local model on Kaggle

The same training as the Colab notebook, from the data already prepared there (`prepared.zip`: `ml/data` + `ml/work`).
Kaggle gives about 30 GPU hours a week, and a notebook saved with **Save Version → Save & Run All** runs by itself in
the background for up to 12 hours, so nobody has to keep the page open.

Before running: **Settings → Accelerator → GPU T4 x2** (or P100), **Settings → Internet → on** (both need a phone
number verified on the Kaggle account). Share `prepared.zip` on Google Drive as «anyone with the link» (it holds only
public pictures under licences that allow sharing) and put its id below. The model, its checkpoint and the zip end up
in **Output** (`/kaggle/working`). To learn from your own judged prints too, add a zip of KlippEye's data folder
(`klippeye/recordings`, `klippeye/reviews`) with **Add Input → Upload**: keep some prints out of it to measure on.

In [ ]:
DRIVE_ID = ""  # the id in the Drive link of prepared.zip: https://drive.google.com/file/d/<id>/view

In [ ]:
assert DRIVE_ID, "put the Drive id of prepared.zip in DRIVE_ID"
!nvidia-smi -L
# the data goes in /tmp: /kaggle/working keeps only 20 GB and is saved as the notebook's output
!git clone -q https://github.com/BotuPotto/KlippEye -b paste-failures /tmp/KlippEye
%cd /tmp/KlippEye
!pip install -q -r ml/requirements.txt gdown
!gdown -q "$DRIVE_ID" -O /tmp/prepared.zip && unzip -q /tmp/prepared.zip && rm /tmp/prepared.zip && wc -l ml/work/manifest.csv

In [ ]:
# 1.135: your own judged prints (a zip of a KlippEye data folder added with «Add Input → Upload»), cut to the bed box:
# the list of pictures is made again with them; without them the prepared one stays
import glob, os
own = glob.glob("/kaggle/input/**/klippeye/recordings", recursive=True)
if own:
    OWN = os.path.dirname(own[0])
    !python ml/datasets.py --auto ml/data --hard-cases 0.3 --mine models/base --klippeye "$OWN" --out ml/work 2>&1 | tail -25

A cut session loses only the epoch it was on: run the notebook again and it resumes from `ckpt.pt` (keep the previous version's output as this one's input, or copy `ckpt.pt` back to `/kaggle/working`).

In [ ]:
!python ml/train.py --manifest ml/work/manifest.csv --out /kaggle/working/model2 --epochs 15 --workers 4 --checkpoint /kaggle/working/ckpt.pt 2>&1 | grep -v Warning

In [ ]:
!cp ml/data/CREDITS.md /kaggle/working/model2/
!mkdir -p /tmp/pasted40 && ls ml/work/pasted | shuf -n 40 | xargs -I{} cp ml/work/pasted/{} /tmp/pasted40/
!python ml/contact_sheet.py /tmp/pasted40 --out /kaggle/working/model2/pasted-sheet
!cd /kaggle/working && zip -q -r model2.zip model2 && ls -la /kaggle/working